In [1]:
import numpy as np 
import pandas as pd 
import os
import pandas as pd
from scipy import stats
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import make_scorer, accuracy_score
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold

In [3]:
data = pd.read_csv('corporate_rating.csv')

In [4]:
data.head()

,Rating,Name,Symbol,Rating Agency Name,Date,Sector,currentRatio,quickRatio,cashRatio,daysOfSalesOutstanding,...,effectiveTaxRate,freeCashFlowOperatingCashFlowRatio,freeCashFlowPerShare,cashPerShare,companyEquityMultiplier,ebitPerRevenue,enterpriseValueMultiple,operatingCashFlowPerShare,operatingCashFlowSalesRatio,payablesTurnover
0,A,Whirlpool Corporation,WHR,Egan-Jones Ratings Company,11/27/2015,Consumer Durables,0.945894,0.426395,0.099690,44.203245,...,0.202716,0.437551,6.810673,9.809403,4.008012,0.049351,7.057088,15.565438,0.058638,3.906655
1,BBB,Whirlpool Corporation,WHR,Egan-Jones Ratings Company,2/13/2014,Consumer Durables,1.033559,0.498234,0.203120,38.991156,...,0.074155,0.541997,8.625473,17.402270,3.156783,0.048857,6.460618,15.914250,0.067239,4.002846
2,BBB,Whirlpool Corporation,WHR,Fitch Ratings,03/06/2015,Consumer Durables,0.963703,0.451505,0.122099,50.841385,...,0.214529,0.513185,9.693487,13.103448,4.094575,0.044334,10.491970,18.888889,0.074426,3.483510
3,BBB,Whirlpool Corporation,WHR,Fitch Ratings,6/15/2012,Consumer Durables,1.019851,0.510402,0.176116,41.161738,...,1.816667,-0.147170,-1.015625,14.440104,3.630950,-0.012858,4.080741,6.901042,0.028394,4.581150
4,BBB,Whirlpool Corporation,WHR,Standard & Poor's Ratings Services,10/24/2016,Consumer Durables,0.957844,0.495432,0.141608,47.761126,...,0.166966,0.451372,7.135348,14.257556,4.012780,0.053770,8.293505,15.808147,0.058065,3.857790


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2029 entries, 0 to 2028
Data columns (total 31 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Rating                              2029 non-null   object 
 1   Name                                2029 non-null   object 
 2   Symbol                              2029 non-null   object 
 3   Rating Agency Name                  2029 non-null   object 
 4   Date                                2029 non-null   object 
 5   Sector                              2029 non-null   object 
 6   currentRatio                        2029 non-null   float64
 7   quickRatio                          2029 non-null   float64
 8   cashRatio                           2029 non-null   float64
 9   daysOfSalesOutstanding              2029 non-null   float64
 10  netProfitMargin                     2029 non-null   float64
 11  pretaxProfitMargin                  2029 no

In [10]:
print(data['Rating'].value_counts())

Rating
BBB    671
BB     490
A      398
B      302
AA      89
CCC     64
AAA      7
CC       5
C        2
D        1
Name: count, dtype: int64


In [11]:
columns_to_drop = ['Rating', 'Name', 'Symbol', 'Rating Agency Name', 'Date', 'Sector']
X = data.drop(columns=columns_to_drop)
ordinal_mapping = {'AAA': 9, 'AA': 8, 'A': 7, 'BBB': 6, 'BB': 5, 'B': 4, 'CCC': 3, 'CC': 2, 'C': 1, 'D': 0}
data['Rating'] = data['Rating'].map(ordinal_mapping)
y = data['Rating']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,  random_state=42)
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test)

params = {
    'objective': 'multi:softmax',  # Use softmax for multi-class classification
    'num_class': 10,  # Number of unique classes (ordinal labels from 0 to 9)
    'eval_metric': 'mlogloss',  # Multi-class log loss
    'max_depth': 5,
    'eta': 0.1
}

# KFold cross-validation setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validation on the training set
cv_results = xgb.cv(
    params=params,
    dtrain=dtrain,
    num_boost_round=100,
    folds=kf,
    metrics="mlogloss",
    as_pandas=True,
    seed=42
)

# Print cross-validation results
print(cv_results)

# Train the final model on the full training set
bst = xgb.train(params, dtrain, num_boost_round=100)

# Predict on the test set
y_pred = bst.predict(dtest)

# Evaluate accuracy on the test set
accuracy = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {accuracy:.2f}')

    train-mlogloss-mean  train-mlogloss-std  test-mlogloss-mean  \
0              1.518618            0.008566            1.571292   
1              1.455635            0.007852            1.544559   
2              1.400821            0.008296            1.520189   
3              1.350794            0.008853            1.499693   
4              1.305214            0.009185            1.482681   
..                  ...                 ...                 ...   
95             0.263780            0.006946            1.279218   
96             0.260227            0.006968            1.278814   
97             0.256841            0.007209            1.278904   
98             0.253051            0.006698            1.279748   
99             0.250040            0.006497            1.279627   

    test-mlogloss-std  
0            0.051529  
1            0.050290  
2            0.047585  
3            0.046398  
4            0.044907  
..                ...  
95           0.057688  
96 

In [13]:
print(pd.DataFrame(zip(y_test, y_pred)).head(20))

    0    1
0   6  6.0
1   5  5.0
2   7  7.0
3   5  6.0
4   5  7.0
5   6  5.0
6   6  5.0
7   5  5.0
8   6  5.0
9   3  4.0
10  6  5.0
11  6  7.0
12  6  5.0
13  4  4.0
14  6  6.0
15  6  6.0
16  6  7.0
17  8  6.0
18  7  7.0
19  4  7.0


In [14]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test)

# XGBoost parameters for feature importance
params = {
    'objective': 'multi:softmax',  # Use softmax for multi-class classification
    'num_class': 10,  # Number of unique classes (ordinal labels from 0 to 9)
    'eval_metric': 'mlogloss',  # Multi-class log loss
    'max_depth': 5,
    'eta': 0.1,
    'verbosity': 1
}

# Train an initial model
bst = xgb.train(params, dtrain, num_boost_round=100)

# Get feature importance
importance = bst.get_score(importance_type='weight')
importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)
importance_df = pd.DataFrame(importance, columns=['Feature', 'Importance'])

# Print feature importance
print(importance_df)

# Select the top N features
top_features = importance_df['Feature'].head(10).tolist()
X_train_selected = X_train[top_features]
X_test_selected = X_test[top_features]

# Convert selected features to DMatrix
dtrain_selected = xgb.DMatrix(X_train_selected, label=y_train)
dtest_selected = xgb.DMatrix(X_test_selected)

# Train a new model with selected features
bst_selected = xgb.train(params, dtrain_selected, num_boost_round=100)

# Predict on the test set with selected features
y_pred = bst_selected.predict(dtest_selected)

# Evaluate accuracy on the test set
accuracy = accuracy_score(y_test, y_pred)
print(f'Test Accuracy with Selected Features: {accuracy:.2f}')

# Print selected features
print(f'Selected Features: {top_features}')

                               Feature  Importance
0                     effectiveTaxRate       677.0
1              enterpriseValueMultiple       675.0
2                         cashPerShare       672.0
3            operatingCashFlowPerShare       648.0
4               daysOfSalesOutstanding       627.0
5                   fixedAssetTurnover       625.0
6                            debtRatio       595.0
7   freeCashFlowOperatingCashFlowRatio       587.0
8                         currentRatio       584.0
9                            cashRatio       563.0
10         operatingCashFlowSalesRatio       541.0
11                   grossProfitMargin       535.0
12                    payablesTurnover       527.0
13                          quickRatio       523.0
14             returnOnCapitalEmployed       509.0
15                      returnOnAssets       430.0
16                       assetTurnover       410.0
17                freeCashFlowPerShare       390.0
18                     netProfi

In [16]:
data = pd.read_csv('corporate_rating.csv')
ordinal_mapping = {
    'AAA': 'A',
    'AA': 'A',
    'A': 'A',
    'BBB': 'B',
    'BB': 'B',
    'B': 'B',
    'CCC': 'Junk',
    'CC': 'Junk',
    'C': 'Junk',
    'D': 'Junk'
}
columns_to_drop = ['Rating', 'Name', 'Symbol', 'Rating Agency Name', 'Date', 'Sector']
X = data.drop(columns=columns_to_drop)
data['Rating'] = data['Rating'].map(ordinal_mapping)
label_mapping = {'A': 2, 'B': 1, 'Junk': 0}
data['Rating'] = data['Rating'].map(label_mapping)
y = data['Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,  random_state=42)
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test)

params = {
    'objective': 'multi:softmax',  # Use softmax for multi-class classification
    'num_class': 3,  # Number of unique classes (ordinal labels from 0 to 9)
    'eval_metric': 'mlogloss',  # Multi-class log loss
    'max_depth': 5,
    'eta': 0.1
}

# KFold cross-validation setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validation on the training set
cv_results = xgb.cv(
    params=params,
    dtrain=dtrain,
    num_boost_round=100,
    folds=kf,
    metrics="mlogloss",
    as_pandas=True,
    seed=42
)

# Print cross-validation results
print(cv_results)

# Train the final model on the full training set
bst = xgb.train(params, dtrain, num_boost_round=100)

# Predict on the test set
y_pred = bst.predict(dtest)

# Evaluate accuracy on the test set
accuracy = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {accuracy:.2f}')
importance = bst.get_score(importance_type='weight')
importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)
importance_df = pd.DataFrame(importance, columns=['Feature', 'Importance'])
print(importance_df)

    train-mlogloss-mean  train-mlogloss-std  test-mlogloss-mean  \
0              0.654286            0.006171            0.675460   
1              0.618177            0.006009            0.656736   
2              0.587941            0.006181            0.641278   
3              0.560468            0.005357            0.629018   
4              0.535948            0.005090            0.617511   
..                  ...                 ...                 ...   
95             0.100509            0.005686            0.538922   
96             0.099207            0.005601            0.539829   
97             0.097896            0.005604            0.540265   
98             0.096656            0.005576            0.540751   
99             0.095380            0.005412            0.540838   

    test-mlogloss-std  
0            0.026647  
1            0.024939  
2            0.022995  
3            0.022918  
4            0.021695  
..                ...  
95           0.029422  
96 

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Load data
data = pd.read_csv('corporate_rating.csv')

# Define the broader ordinal mapping
ordinal_mapping = {
    'AAA': 'A',
    'AA': 'A',
    'A': 'A',
    'BBB': 'B',
    'BB': 'B',
    'B': 'B',
    'CCC': 'Junk',
    'CC': 'Junk',
    'C': 'Junk',
    'D': 'Junk'
}

# Drop irrelevant columns and map the ratings
columns_to_drop = ['Rating', 'Name', 'Symbol', 'Rating Agency Name', 'Date', 'Sector']
X = data.drop(columns=columns_to_drop)
data['Rating'] = data['Rating'].map(ordinal_mapping)

# Convert broader categories to integers
label_mapping = {'A': 2, 'B': 1, 'Junk': 0}
data['Rating'] = data['Rating'].map(label_mapping)

# Split into features and target
X = X.fillna(0)  # Handle any missing values in feature columns
y = data['Rating']

# Split into train and test sets with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train KNN model
knn = KNeighborsClassifier(n_neighbors=5)  # You can tune the number of neighbors
knn.fit(X_train_scaled, y_train)

# Predict on the test set
y_pred = knn.predict(X_test_scaled)

# Evaluate accuracy on the test set
accuracy = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {accuracy:.2f}')

Test Accuracy: 0.74


In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Load data
data = pd.read_csv('corporate_rating.csv')

# Define the broader ordinal mapping
ordinal_mapping = {
    'AAA': 'A',
    'AA': 'A',
    'A': 'A',
    'BBB': 'B',
    'BB': 'B',
    'B': 'B',
    'CCC': 'Junk',
    'CC': 'Junk',
    'C': 'Junk',
    'D': 'Junk'
}

# Drop irrelevant columns and map the ratings
columns_to_drop = ['Rating', 'Name', 'Symbol', 'Rating Agency Name', 'Date', 'Sector']
X = data.drop(columns=columns_to_drop)
ordinal_mapping = {'AAA': 9, 'AA': 8, 'A': 7, 'BBB': 6, 'BB': 5, 'B': 4, 'CCC': 3, 'CC': 2, 'C': 1, 'D': 0}
data['Rating'] = data['Rating'].map(ordinal_mapping)
y = data['Rating']

# Split into features and target
X = X.fillna(0)  # Handle any missing values in feature columns


# Split into train and test sets with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train KNN model
knn = KNeighborsClassifier(n_neighbors=5)  # You can tune the number of neighbors
knn.fit(X_train_scaled, y_train)

# Predict on the test set
y_pred = knn.predict(X_test_scaled)

# Evaluate accuracy on the test set
accuracy = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {accuracy:.2f}')

Test Accuracy: 0.34
